# Spotify Datamining Notebook
Dit notebook verzamelt en analyseert muzieknummers via de Spotify API.
Het doel is om audiofeatures (tempo, energy, valence, acousticness, ...) te extraheren
voor het bouwen van wetenschappelijk onderbouwde playlists.

In [3]:
!pip install spotipy openpyxl keyring


In [4]:
import os
import time
import pandas as pd
import keyring
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from spotipy.exceptions import SpotifyException


Momenteel is het niet mogelijk om een app te maken via de spotify web API, hieronder wordt een structuur weergeven die gebruik zou maken van de app. Er is nog geen datum vrijgegeven wanneer de app terug toegankelijk is vandaar het on hold gegeven. Kan later in gebruik gesteld worden moest de app vrijgeven worden. Momenteel focus op bestaande datasets met audiofeatures om via die weg, nummers en hun kenmerken te verkrijgen voor het samenstellen van playlists


### Spotify authenticatie - keyring

In [ ]:
client_id = keyring.get_password("spotify", "client_id")
client_secret = keyring.get_password("spotify", "client_secret")

if client_id is None or client_secret is None:
    raise ValueError("Spotify credentials niet gevonden in keyring. Voeg ze eerst toe.")

auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(auth_manager=auth_manager)


### Wrapper voor API‑calls (rate‑limit vriendelijk)

In [ ]:
def safe_call(func, *args, **kwargs):
    """
    Wrapper rond Spotify API-calls.
    - Ondervangt error 429 (rate limit) en wacht 'Retry-After' seconden.
    - Voor andere fouten: raise opnieuw.
    """
    while True:
        try:
            return func(*args, **kwargs)
        except SpotifyException as e:
            if e.http_status == 429:
                retry_after = int(e.headers.get("Retry-After", 5))
                print(f"Rate limit bereikt, wachten {retry_after} seconden...")
                time.sleep(retry_after)
            else:
                raise


### Helperfuncties voor playlists en features

#### Tracks uit een playlist ophalen

In [ ]:
def get_playlist_tracks(playlist_id, max_tracks=None):
    """
    Haalt alle track IDs op uit een Spotify playlist.
    - playlist_id: ID of URI van de playlist
    - max_tracks: optioneel maximum aantal tracks om op te halen (voor veiligheid)
    """
    results = safe_call(sp.playlist_tracks, playlist_id)
    tracks = results["items"]

    while results["next"] and (max_tracks is None or len(tracks) < max_tracks):
        results = safe_call(sp.next, results)
        tracks.extend(results["items"])

    # Filter lege entries en beperk indien max_tracks is gezet
    tracks = [t for t in tracks if t.get("track")]
    if max_tracks is not None:
        tracks = tracks[:max_tracks]

    track_ids = [t["track"]["id"] for t in tracks]
    return track_ids


#### Kenmerken en metadata per track

In [ ]:
def get_track_features(track_id):
    """
    Haalt metadata en audiofeatures op voor één track.
    Geeft een dict terug met de belangrijkste velden.
    """
    features_list = safe_call(sp.audio_features, [track_id])
    features = features_list[0]
    meta = safe_call(sp.track, track_id)

    return {
        "id": track_id,
        "name": meta["name"],
        "artist": meta["artists"][0]["name"],
        "album": meta["album"]["name"],
        "tempo": features["tempo"],
        "energy": features["energy"],
        "valence": features["valence"],
        "acousticness": features["acousticness"],
        "instrumentalness": features["instrumentalness"],
        "danceability": features["danceability"],
        "liveness": features["liveness"],
        "loudness": features["loudness"],
        "duration_ms": features["duration_ms"]
    }


#### Analyse van meerdere tracks in één keer

In [ ]:
def analyze_tracks(track_ids, max_calls=None):
    """
    Maakt een DataFrame met audiofeatures voor een lijst track IDs.
    - max_calls: optioneel, beperkt het aantal tracks dat effectief wordt opgehaald.
    """
    data = []
    for i, tid in enumerate(track_ids):
        if max_calls is not None and i >= max_calls:
            print(f"Maximaal aantal calls ({max_calls}) bereikt, stoppen met analyseren.")
            break
        info = get_track_features(tid)
        data.append(info)
    return pd.DataFrame(data)


### Nummers analyseren uit bestaande playlist

In [ ]:
playlist_id = "37i9dQZF1DX3PIPIT6lEg5"  # voorbeeld: 'Peaceful Piano' (zonder 'spotify:playlist:' prefix)

track_ids = get_playlist_tracks(playlist_id, max_tracks=200)
print(f"Aantal tracks opgehaald: {len(track_ids)}")

df_playlist = analyze_tracks(track_ids, max_calls=200)
df_playlist.head()


### Gerichte zoekopdracht naar nummers op basis van kenmerken

In [ ]:
results = safe_call(
    sp.recommendations,
    seed_genres=["ambient"],
    limit=50,
    min_tempo=50,
    max_tempo=70,
    max_energy=0.4,
    min_acousticness=0.5
)

rec_ids = [t["id"] for t in results["tracks"]]
df_rec = analyze_tracks(rec_ids, max_calls=50)
df_rec.head()


### Data combineren en cleaning

In [ ]:
df_all = pd.concat([df_playlist, df_rec], ignore_index=True)
df_all.drop_duplicates(subset="id", inplace=True)
df_all.shape


### Export naar Excel of CSV

In [ ]:
df_all.to_excel("spotify_datamining.xlsx", index=False)
df_all.to_csv("spotify_datamining.csv", index=False)
